In [0]:
from pyspark.sql.functions import col, trim, to_date, expr

In [0]:
display(spark.table("workspace.recruitment.bronze_claims").limit(10))

In [0]:
spark.table("workspace.recruitment.bronze_claims").printSchema()

In [0]:
silver_df = (
    spark.readStream.table("workspace.recruitment.bronze_claims")
    .select(
        # Customer / policy
        expr("try_cast(months_as_customer as int)").alias("months_as_customer"),
        expr("try_cast(age as int)").alias("age"),
        col("policy_number").alias("policy_number"),
        to_date(trim(col("policy_bind_date")), "yyyy-MM-dd").alias("policy_bind_date"),
        trim(col("policy_state")).alias("policy_state"),
        trim(col("policy_csl")).alias("policy_csl"),
        expr("try_cast(policy_deductable as int)").alias("policy_deductable"),
        expr("try_cast(policy_annual_premium as decimal(12,2))").alias("policy_annual_premium"),
        expr("try_cast(umbrella_limit as bigint)").alias("umbrella_limit"),
        trim(col("insured_zip")).alias("insured_zip"),

        # Insured
        trim(col("insured_sex")).alias("insured_sex"),
        trim(col("insured_education_level")).alias("insured_education_level"),
        trim(col("insured_occupation")).alias("insured_occupation"),
        trim(col("insured_hobbies")).alias("insured_hobbies"),
        trim(col("insured_relationship")).alias("insured_relationship"),

        # Financial
        expr("try_cast(`capital-gains` as bigint)").alias("capital-gains"),
        expr("try_cast(`capital-loss` as bigint)").alias("capital-loss"),

        # Incident
        to_date(trim(col("incident_date")), "yyyy-MM-dd").alias("incident_date"),
        trim(col("incident_type")).alias("incident_type"),
        trim(col("collision_type")).alias("collision_type"),
        trim(col("incident_severity")).alias("incident_severity"),
        trim(col("authorities_contacted")).alias("authorities_contacted"),
        trim(col("incident_state")).alias("incident_state"),
        trim(col("incident_city")).alias("incident_city"),
        trim(col("incident_location")).alias("incident_location"),
        expr("try_cast(incident_hour_of_the_day as int)").alias("incident_hour_of_the_day"),

        # Incident details
        expr("try_cast(number_of_vehicles_involved as int)").alias("number_of_vehicles_involved"),
        trim(col("property_damage")).alias("property_damage"),
        expr("try_cast(bodily_injuries as int)").alias("bodily_injuries"),
        expr("try_cast(witnesses as int)").alias("witnesses"),
        trim(col("police_report_available")).alias("police_report_available"),

        # Claim amounts
        expr("try_cast(total_claim_amount as decimal(14,2))").alias("total_claim_amount"),
        expr("try_cast(injury_claim as decimal(14,2))").alias("injury_claim"),
        expr("try_cast(property_claim as decimal(14,2))").alias("property_claim"),
        expr("try_cast(vehicle_claim as decimal(14,2))").alias("vehicle_claim"),

        # Vehicle
        trim(col("auto_make")).alias("auto_make"),
        trim(col("auto_model")).alias("auto_model"),
        expr("try_cast(auto_year as int)").alias("auto_year"),

        # Target
        trim(col("fraud_reported")).alias("fraud_reported"),

        # Metadata
        expr("try_cast(timestamp as timestamp)").alias("timestamp"),
        expr("try_cast(_ingestion_timestamp as timestamp)").alias("_ingestion_timestamp"),
        col("_source_file").alias("_source_file"),
    )
)

In [0]:
silver_df.printSchema()

In [0]:
from pyspark.sql.functions import col, when, lit, concat_ws

validated_df = (
    silver_df
    .withColumn(
        "validation_error",
        concat_ws(
            "; ",
            when(col("policy_number").isNull(), lit("missing_policy_number")),
            when(col("policy_bind_date").isNull(), lit("missing_policy_bind_date")),
            when(col("incident_date").isNull(), lit("missing_incident_date")),
            when(col("total_claim_amount").isNull(), lit("missing_total_claim_amount")),

            when(
                (col("age") < 18) | (col("age") > 100),
                lit("invalid_age")
            ),

            when(
                col("months_as_customer") < 0,
                lit("invalid_customer_tenure")
            ),

            when(
                col("policy_annual_premium") < 0,
                lit("negative_premium")
            ),

            when(
                col("total_claim_amount") < 0,
                lit("negative_claim_amount")
            ),

            when(
                col("number_of_vehicles_involved") < 1,
                lit("invalid_vehicle_count")
            ),

            when(
                (col("incident_hour_of_the_day") < 0) |
                (col("incident_hour_of_the_day") > 23),
                lit("invalid_incident_hour")
            ),

            when(
                col("incident_date") < col("policy_bind_date"),
                lit("incident_before_policy")
            ),

            when(
                col("fraud_reported").isin("Y", "N") == False,
                lit("invalid_fraud_flag")
            ),

            when(
                col("property_damage").isin("YES", "NO") == False,
                lit("invalid_property_damage")
            ),

            when(
                col("police_report_available").isin("YES", "NO") == False,
                lit("invalid_police_report")
            )
        )
    )
)

In [0]:
validated_df = validated_df.withColumn(
    "is_valid",
    col("validation_error") == ""
)

In [0]:
%skip
temp_table = "recruitment.silver_temp"
silver_checkpoint = "/Volumes/workspace/recruitment/policies/checkpoints/silver_temp"

query = (
    validated_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", silver_checkpoint)
    .trigger(availableNow=True)
    .toTable(temp_table)
)

display(
    spark.table("recruitment.silver_temp")
)

In [0]:
silver_valid_df = (
    validated_df
    .filter(col("is_valid"))
    .drop("validation_error", "is_valid")
)

silver_quarantine_df = (
    validated_df
    .filter(~col("is_valid"))
)

In [0]:
silver_table = "recruitment.silver_claims"
silver_checkpoint = "/Volumes/workspace/recruitment/policies/checkpoints/silver_claims"

query = (
    silver_valid_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", silver_checkpoint)
    .trigger(availableNow=True)
    .toTable(silver_table)
)

quarantine_table = "recruitment.silver_claims_quarantine"
quarantine_checkpoint = (
    "/Volumes/workspace/recruitment/policies/checkpoints/silver_claims_quarantine"
)

quarantine_query = (
    silver_quarantine_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", quarantine_checkpoint)
    .trigger(availableNow=True)
    .toTable(quarantine_table)
)

In [0]:
display(
    spark.table("recruitment.silver_claims_quarantine")
)

display(
    spark.table("recruitment.silver_claims_quarantine")
    .groupBy("validation_error")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
bronze_count = spark.table("recruitment.bronze_claims").count()

silver_count = spark.table("recruitment.silver_claims").count()

quarantine_count = spark.table(
    "recruitment.silver_claims_quarantine"
).count()

print("Bronze:", bronze_count)
print("Silver:", silver_count)
print("Quarantine:", quarantine_count)
print("Silver + quarantine:", silver_count + quarantine_count)